# Week 15: Capstone & Interview Prep

**Lab companion to [week_15.md](../agenda/week_15.md)**

This final lab covers:
1. Capstone project structure
2. Common interview questions
3. Code challenges
4. Project review checklist

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

print("✓ Ready for the final lab!")

## 1. Interview Questions & Answers

Common AI/LLM Engineering interview questions:

In [ ]:
interview_questions = {
    "What is RAG and why use it?": """
    RAG (Retrieval-Augmented Generation) combines retrieval with LLM generation.

    Why use it:
    - Reduces hallucinations by grounding responses in real data
    - Enables working with private/recent data not in training
    - More cost-effective than fine-tuning for knowledge updates
    - Provides source citations for transparency

    Key components: Document chunking, embedding, vector search, generation
    """,

    "How do you prevent prompt injection?": """
    1. Input validation and sanitization
    2. Separate user input from system instructions
    3. Use structured formats (JSON mode)
    4. LLM-based injection detection
    5. Output filtering
    6. Rate limiting
    7. Content moderation APIs
    """,

    "What's the difference between embeddings and fine-tuning?": """
    Embeddings (RAG):
    - External knowledge retrieval
    - Easy to update knowledge
    - No model training needed
    - Good for facts and documents

    Fine-tuning:
    - Changes model behavior/style
    - Requires training data and compute
    - Better for specific formats/tasks
    - Knowledge baked into model
    """,

    "How do you evaluate LLM outputs?": """
    1. Retrieval metrics: Precision, Recall, MRR
    2. Generation metrics:
       - LLM-as-judge (accuracy, relevance, coherence)
       - Human evaluation
       - Task-specific metrics (BLEU, ROUGE for summarization)
    3. A/B testing different configurations
    4. Ground-truth comparison on golden datasets
    """,

    "How do you handle rate limits in production?": """
    1. Token bucket rate limiting
    2. Exponential backoff with jitter
    3. Request queuing
    4. Multiple API keys rotation
    5. Fallback to different models
    6. Caching frequent responses
    7. Circuit breaker pattern
    """
}

# Display
for q, a in interview_questions.items():
    print(f"Q: {q}")
    print(f"A: {a.strip()}")
    print("=" * 60)

## 2. Coding Challenge: Build a Mini RAG

In [ ]:
# CHALLENGE: Build a complete RAG system from scratch
# Time: 30-45 minutes

import numpy as np

class MiniRAG:
    """
    Build a RAG system with:
    - Document storage
    - Embedding-based retrieval
    - Answer generation with citations
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.client = OpenAI()
        self.model = model
        self.documents = []  # List of {text, embedding, metadata}

    def add_document(self, text: str, metadata: dict = None):
        """Add a document to the knowledge base."""
        embedding = self._get_embedding(text)
        self.documents.append({
            "text": text,
            "embedding": embedding,
            "metadata": metadata or {}
        })

    def query(self, question: str, top_k: int = 3) -> dict:
        """Answer a question using RAG."""
        # Retrieve
        relevant = self._retrieve(question, top_k)

        # Generate
        context = "\n\n".join([f"[{i+1}] {d['text']}" for i, d in enumerate(relevant)])

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": "Answer based on context. Cite sources with [N]."},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
            ]
        )

        return {
            "answer": response.choices[0].message.content,
            "sources": [d["text"][:50] + "..." for d in relevant]
        }

    def _get_embedding(self, text: str) -> list:
        response = self.client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return response.data[0].embedding

    def _retrieve(self, query: str, top_k: int) -> list:
        query_emb = self._get_embedding(query)

        # Calculate similarities
        scores = []
        for doc in self.documents:
            sim = self._cosine_similarity(query_emb, doc["embedding"])
            scores.append((doc, sim))

        # Sort and return top_k
        scores.sort(key=lambda x: x[1], reverse=True)
        return [doc for doc, _ in scores[:top_k]]

    def _cosine_similarity(self, a: list, b: list) -> float:
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Test
rag = MiniRAG()

# Add documents
docs = [
    "Python was created by Guido van Rossum and released in 1991.",
    "JavaScript was created by Brendan Eich at Netscape in 1995.",
    "Machine learning is a subset of AI that learns from data.",
    "Deep learning uses neural networks with multiple layers."
]

for doc in docs:
    rag.add_document(doc)

# Query
result = rag.query("Who created Python and when?")
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")

## 3. Coding Challenge: Implement Retry Logic

In [ ]:
# CHALLENGE: Implement retry with exponential backoff
# Time: 15 minutes

import time
import random
from functools import wraps

def with_retry(max_retries: int = 3, base_delay: float = 1.0):
    """Decorator that adds retry with exponential backoff."""

    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_error = None

            for attempt in range(max_retries + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_error = e

                    if attempt < max_retries:
                        delay = base_delay * (2 ** attempt) * (0.5 + random.random())
                        print(f"  Attempt {attempt + 1} failed, retrying in {delay:.1f}s...")
                        time.sleep(delay)

            raise last_error

        return wrapper
    return decorator

# Test
@with_retry(max_retries=3)
def flaky_function():
    if random.random() < 0.6:
        raise Exception("Random failure")
    return "Success!"

try:
    result = flaky_function()
    print(f"Result: {result}")
except Exception as e:
    print(f"Final failure: {e}")

## 4. Capstone Project Checklist

In [ ]:
capstone_checklist = """
## Capstone Project Requirements Checklist

### Core Features ✓
- [ ] LLM integration (OpenAI API)
- [ ] RAG implementation with vector search
- [ ] Document ingestion pipeline
- [ ] Query interface (API or UI)

### Production Quality ✓
- [ ] Error handling & retry logic
- [ ] Rate limiting
- [ ] Input validation
- [ ] Security (prompt injection prevention)
- [ ] Logging & monitoring

### Documentation ✓
- [ ] README with setup instructions
- [ ] API documentation
- [ ] Architecture diagram
- [ ] Cost estimation

### Testing ✓
- [ ] Unit tests for key functions
- [ ] Evaluation metrics defined
- [ ] Test dataset with golden answers

### Bonus ✓
- [ ] Multi-modal support
- [ ] Agent/tool calling
- [ ] Docker deployment
- [ ] CI/CD pipeline
"""

print(capstone_checklist)

## 5. Quick Reference: Key Patterns

In [ ]:
# Pattern 1: Basic Chat
def basic_chat(message: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": message}]
    )
    return response.choices[0].message.content

# Pattern 2: Structured Output
from pydantic import BaseModel

class Output(BaseModel):
    answer: str
    confidence: float

def structured_output(message: str) -> Output:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": message}],
        response_format={"type": "json_object"}
    )
    import json
    data = json.loads(response.choices[0].message.content)
    return Output(**data)

# Pattern 3: Embedding
def get_embedding(text: str) -> list:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# Pattern 4: Tool Calling
def call_with_tools(message: str, tools: list) -> dict:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": message}],
        tools=tools
    )
    return response.choices[0].message

print("Key patterns ready for reference!")

## 🎉 Congratulations!

You've completed the 16-week AI Engineering curriculum.

### Key Skills Acquired:
- ✅ OpenAI API mastery
- ✅ RAG implementation
- ✅ Embeddings & vector search
- ✅ Prompt engineering
- ✅ Tool calling & agents
- ✅ Security & production hardening
- ✅ Evaluation & testing
- ✅ Deployment

### Next Steps:
1. Complete your capstone project
2. Build a portfolio of AI projects
3. Stay updated with new models and techniques
4. Practice interview questions
5. Contribute to open-source AI projects

Good luck with your AI engineering journey! 🚀